# 🎯 Face Accessory Detector — Colab Training & Serving Runner
> MobileNetV3-Small multi-label classifier on face crops
> Two-Stage Pipeline: RetinaFace (buffalo_s) → MobileNetV3-Small

**Instructions:**
1. Connect to GPU Runtime: **Runtime → Change runtime type → T4 GPU**.
2. This notebook is a runner. It pulls your modular code files from Git (or Google Drive) and executes them directly. This makes training extremely clean, robust, and version-controlled.

**Classes:** glasses_clear, glasses_tinted, helmet_bike, helmet_hard, cap_hat, beanie, scarf_muffler, face_mask, face_shield, balaclava

## Step 0 — Mount Google Drive & Set Directories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

# We run inside our Google Drive folder
BASE_DIR = '/content/drive/MyDrive/face_accessory_detector'
os.makedirs(BASE_DIR, exist_ok=True)
%cd {BASE_DIR}

print('✓ Working directory set to:', BASE_DIR)

## Step 1 — Clone or Pull the Git Repository
Replace `<YOUR_GIT_REPO_URL>` with your repository's URL (e.g. on GitHub, GitLab, or Bitbucket).

In [ ]:
REPO_URL = 'YOUR_GIT_REPO_URL_HERE' # e.g. https://github.com/username/repo.git
REPO_NAME = 'occlusion_model_train'

if not os.path.exists(REPO_NAME):
    print(f'Cloning repository {REPO_URL}...')
    !git clone {REPO_URL} {REPO_NAME}
else:
    print('Repository directory exists. Pulling latest updates...')
    %cd {REPO_NAME}
    !git pull
    %cd ..

# Append the repository directory to Python search path
sys.path.append(os.path.join(BASE_DIR, REPO_NAME))
%cd {BASE_DIR}/{REPO_NAME}

## Step 2 — Install Dependencies

In [ ]:
%%capture
!pip install -r requirements.txt
print('✓ All dependencies installed successfully.')

## Step 3 — Data Acquisition
Downloads FiftyOne OpenImages v7, Roboflow public datasets, Hugging Face CelebA dataset, and LFW negatives. This handles caching automatically.

In [ ]:
# Set Roboflow key if downloading Roboflow datasets
# %env ROBOFLOW_API_KEY=your_key_here

!python src/data/download.py --config config.yaml

## Step 4 — Preprocessing, Annotations & Splits
Extracts face crops, merges annotations, splits data 80/10/10, and runs CLIP zero-shot labeling on unlabeled crops.

In [ ]:
# Run with --autolabel flag to expand labels for rare classes using CLIP zero-shot
!python src/data/annotate.py --config config.yaml --autolabel

## Step 5 — Train Model
Trains MobileNetV3-Small using mixed precision (FP16), weighted BCE, and Cosine Annealing learning rate schedule.

In [ ]:
# Model supports auto-resume if training is interrupted
!python src/train.py --config config.yaml

## Step 6 — Evaluate Classifier
Tests model performance on the test set split, computes F1-optimized per-class thresholds, and prints classification report.

In [ ]:
!python src/evaluate.py --config config.yaml --model outputs/checkpoints/best.pth

## Step 7 — Model Export & Latency Benchmarks
Exports checkpoints to simplified ONNX graphs and compiles to optimized TensorRT engine. Runs latency comparisons.

In [ ]:
!python src/export.py --config config.yaml --model outputs/checkpoints/best.pth

## Step 8 — Quick Interactive Visual Test
Upload a custom test image and see what accessories the two-stage pipeline detects!

In [ ]:
from google.colab import files
import PIL.Image
import yaml

uploaded = files.upload()

# Import our pipeline module directly
from src.models.pipeline import AccessoryPipeline

# Load F1-optimized thresholds calculated during evaluation if available
thresholds = None
thresh_path = 'outputs/optimal_thresholds.yaml'
if os.path.exists(thresh_path):
    with open(thresh_path) as f:
        thresholds = yaml.safe_load(f)
    print('✓ Loaded F1-optimized thresholds from evaluate.py')

# Initialize pipeline
pipeline = AccessoryPipeline(
    classifier_path='outputs/accessor_classifier.onnx',
    face_model='buffalo_s',
    device='cuda',
    thresholds=thresholds
)

for fname in uploaded:
    img = cv2.imread(fname)
    if img is None: continue
    
    detections = pipeline.infer(img)
    print(f'\nFound {len(detections)} face(s) in {fname}:')
    
    for idx, det in enumerate(detections):
        print(f'\n  Face {idx+1} (Detection score: {det.confidence:.2f}):')
        
        # Display detected bboxes
        x1, y1, x2, y2 = det.bbox
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Sort and display probability bar charts
        for cls, prob in sorted(det.accessories.items(), key=lambda x: -x[1]):
            bar = '█' * int(prob * 20)
            tag = ' ✓ DETECTED' if cls in det.detected_classes else ''
            print(f'    {cls:<18} {prob:.3f} {bar}{tag}')
            
    # Show processed output frame
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    display(PIL.Image.fromarray(img_rgb))